In [1]:
import pandas as pd

df = pd.read_csv('unified_claims_model_dataset.csv')

print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)

print("\nMissing Values Summary:\n", df.isnull().sum())

df.head(10)


Dataset Shape: (7500, 18)

Column Names: ['patient_id', 'age_group', 'sex', 'region', 'provider_id', 'speciality', 'npi', 'procedure_code', 'procedure_descrption', 'category', 'date_of_service', 'billed_amount', 'paid_amount', 'adjudication_status', 'Provider Organization Name (Legal Business Name)', 'Provider Last Name (Legal Name)', 'Provider First Name', 'is_fake_npi']

Data Types:
 patient_id                                           object
age_group                                            object
sex                                                  object
region                                               object
provider_id                                         float64
speciality                                           object
npi                                                 float64
procedure_code                                       object
procedure_descrption                                 object
category                                             object
date_of_ser

,patient_id,age_group,sex,region,provider_id,speciality,npi,procedure_code,procedure_descrption,category,date_of_service,billed_amount,paid_amount,adjudication_status,Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,is_fake_npi
0,PT100000,0-17,M,South,9.639473e+09,Podiatry,9.639473e+09,11721,Debridement nail,Procedure,2023-10-13,5944.48,5880.39,Paid,NaN,ROSENBLOOM,FRANK,0
1,PT100001,35-49,M,North,6.303021e+09,Plastic Surgery,NaN,15734,Scar revision,Procedure,2024-03-23,2014.75,1926.35,Paid,NaN,MINOR,SUMMER,0
2,PT100002,18-34,F,North,9.670336e+09,Infectious Disease,9.670336e+09,96372,Therapeutic injection,Procedure,2023-11-23,NaN,5168.72,Paid,NaN,RICHERT,EMMA,0
3,PT100003,0-17,M,East,3.381396e+09,ENT,3.381396e+09,92511,Speech audiometry,Diagnostic,2024-12-05,502.79,NaN,Partially Paid,FULL CIRCLE RECOVERY OF CIRCLEVILLE,NaN,NaN,0
4,PT100004,65+,M,Central,7.892771e+09,Infectious Disease,7.892771e+09,87070,NaN,Lab,2024-08-23,418.11,NaN,Paid,NaN,TREVISAN,MARIE,0
5,PT100005,50-64,M,South,7.929459e+09,Psychiatry,7.929459e+09,90791,Psychiatric diagnostic eval,Evaluation & Management,2024-08-15,394.83,278.96,Partially Paid,NaN,MASON,SHAMEKA,0
6,PT100006,18-34,F,East,9.581991e+09,Infectious Disease,9.581991e+09,96372,Therapeutic injection,Procedure,2023-11-30,3087.41,2861.10,Paid,NaN,CRUZ,ANDREW,0
7,PT100007,35-49,M,South,8.530855e+09,Orthopedics,8.530855e+09,20610,"Arthrocentesis, small joint",Procedure,2023-12-01,3805.63,3777.24,Paid,NaN,PARE,RONNIESHA,0
8,PT100008,35-49,F,North,1.445200e+09,Cardiology,8.001668e+09,93306,Echocardiogram,Imaging,2024-04-14,666.35,253.76,Partially Paid,NaN,BYER,JENNIFER,1
9,PT100009,18-34,F,East,6.754503e+09,Podiatry,6.754503e+09,11721,Debridement nail,Procedure,2024-03-11,5312.42,5267.02,Paid,NaN,BENTSUR,SHARONA,0


In [2]:
import hashlib
# Helper function for hashing IDs
def hash_id(x):
    return hashlib.sha256(str(x).encode()).hexdigest()


In [4]:

# Create dim_patient from original dataset
dim_patient = df[['patient_id', 'age_group', 'sex', 'region']].drop_duplicates().reset_index(drop=True)

# Add surrogate key
dim_patient['patient_sk'] = range(1, len(dim_patient) + 1)

# Hash patient_id for privacy
def hash_id(x):
    return hash(str(x))

dim_patient['patient_id_hashed'] = dim_patient['patient_id'].apply(hash_id)
# Define mapping for age groups to descriptive categories
age_mapping = {
    '0-17': 'Child',
    '18-34': 'Young Adult',
    '35-49': 'Adult',
    '50-64': 'Middle Age',
    '65+': 'Senior'
}

# Replace age_group values with descriptive categories in the same column
df['age_group'] = df['age_group'].map(age_mapping)


# Select final columns
dim_patient = dim_patient[['patient_sk', 'patient_id_hashed', 'age_group', 'sex', 'region']]

# Preview
print(dim_patient.head())


   patient_sk    patient_id_hashed    age_group sex   region
0           1  9008202424718923229        Child   M    South
1           2  3881683668997963159        Adult   M    North
2           3 -3633048590672850712  Young Adult   F    North
3           4  4433913897963147793        Child   M     East
4           5 -5034808169511523917       Senior   M  Central


In [5]:

dim_provider = df[['provider_id', 'speciality', 'npi']].drop_duplicates().reset_index(drop=True)
dim_provider['provider_sk'] = range(1, len(dim_provider) + 1)
dim_provider['provider_id'] = dim_provider['provider_id'].fillna(-1)
dim_provider['npi'] = dim_provider['npi'].fillna(-1)
dim_provider = dim_provider[['provider_sk', 'provider_id', 'speciality', 'npi']]

default_provider_sk = 0
if -1 in dim_provider['provider_id'].values:
    dim_provider = pd.concat([
        pd.DataFrame([[default_provider_sk, -1, 'Unknown', -1]], columns=['provider_sk', 'provider_id', 'speciality', 'npi']),
        dim_provider
    ], ignore_index=True)




dim_provider.head()

,provider_sk,provider_id,speciality,npi
0,0,-1.000000e+00,Unknown,-1.000000e+00
1,1,9.639473e+09,Podiatry,9.639473e+09
2,2,6.303021e+09,Plastic Surgery,-1.000000e+00
3,3,9.670336e+09,Infectious Disease,9.670336e+09
4,4,3.381396e+09,ENT,3.381396e+09


In [10]:

import pandas as pd
import hashlib

# Load main dataset
claims_df = pd.read_csv("unified_claims_model_dataset.csv")

# Load CPT and ICD reference files
cpt_df = pd.read_csv('cpt4.csv')  # Replace with actual file name
icd_df = pd.read_csv('icd10_mapped_output.csv')  # Replace with actual file name

# Check column names
print("CPT columns:", cpt_df.columns)
print("ICD columns:", icd_df.columns)

# Create mapping dictionaries using correct column names
cpt_mapping = dict(zip(cpt_df['com.medigy.persist.reference.type.clincial.CPT.code'].astype(str),
                       cpt_df['label']))
icd_mapping = dict(zip(icd_df['ICD-10 Code'].astype(str),
                       icd_df['ICD Description']))

# Extract relevant columns for dim_procedure
procedure_df = claims_df[['procedure_code', 'procedure_descrption', 'category']].copy()

# Drop duplicates
procedure_df = procedure_df.drop_duplicates(subset=['procedure_code'])

# Fill missing descriptions using CPT or ICD mapping
procedure_df['procedure_descrption'] = procedure_df.apply(
    lambda row: row['procedure_descrption'] if pd.notnull(row['procedure_descrption'])
    else cpt_mapping.get(str(row['procedure_code']),
                         icd_mapping.get(str(row['procedure_code']), 'Unknown Procedure')),
    axis=1
)

# Fill missing category
procedure_df['category'] = procedure_df['category'].fillna('Uncategorized')

# Generate surrogate key
procedure_df['proc_sk'] = procedure_df['procedure_code'].apply(
    lambda x: int(hashlib.sha256(str(x).encode()).hexdigest(), 16) % (10**8)
)

# Final table
dim_procedure = procedure_df[['proc_sk', 'procedure_code', 'procedure_descrption', 'category']]

# Validate uniqueness
assert dim_procedure['proc_sk'].is_unique, "Surrogate keys are not unique!"

# Save
dim_procedure.to_csv('dim_procedure.csv', index=False)

print("dim_procedure table created successfully!")
print(dim_procedure.head())


CPT columns: Index(['com.medigy.persist.reference.type.clincial.CPT.code', 'label'], dtype='object')
ICD columns: Index(['Diagnosis', 'ICD-10 Code', 'ICD Description', 'Similarity Score',
       'Justification', 'Alternative Suggestions', 'Needs Review'],
      dtype='object')
dim_procedure table created successfully!
    proc_sk procedure_code      procedure_descrption    category
0  74476075          11721          Debridement nail   Procedure
1  69893948          15734             Scar revision   Procedure
2  35867643          96372     Therapeutic injection   Procedure
3  95044104          92511         Speech audiometry  Diagnostic
4  21477593          87070  Culture, bacteria, other         Lab


In [15]:

# # Map patient_sk from dim_patient
# patient_map = dict(zip(df['patient_id'], dim_patient['patient_sk']))
# fact_claim['patient_sk'] = fact_claim['patient_id'].map(patient_map)

# # Map provider_sk from dim_provider
# provider_map = dict(zip(dim_provider['provider_id'], dim_provider['provider_sk']))
# fact_claim['provider_sk'] = fact_claim['provider_id'].map(provider_map)

# # Map proc_sk from dim_procedure
# proc_map = dict(zip(dim_procedure['procedure_code'], dim_procedure['proc_sk']))
# fact_claim['proc_sk'] = fact_claim['procedure_code'].map(proc_map)

# # Add claim_sk
# fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)
df.head()

,patient_id,age_group,sex,region,provider_id,speciality,npi,procedure_code,procedure_descrption,category,date_of_service,billed_amount,paid_amount,adjudication_status,Provider Organization Name (Legal Business Name),Provider Last Name (Legal Name),Provider First Name,is_fake_npi
0,PT100000,0-17,M,South,9.639473e+09,Podiatry,9.639473e+09,11721,Debridement nail,Procedure,2023-10-13,5944.48,5880.39,Paid,NaN,ROSENBLOOM,FRANK,0
1,PT100001,35-49,M,North,6.303021e+09,Plastic Surgery,NaN,15734,Scar revision,Procedure,2024-03-23,2014.75,1926.35,Paid,NaN,MINOR,SUMMER,0
2,PT100002,18-34,F,North,9.670336e+09,Infectious Disease,9.670336e+09,96372,Therapeutic injection,Procedure,2023-11-23,NaN,5168.72,Paid,NaN,RICHERT,EMMA,0
3,PT100003,0-17,M,East,3.381396e+09,ENT,3.381396e+09,92511,Speech audiometry,Diagnostic,2024-12-05,502.79,NaN,Partially Paid,FULL CIRCLE RECOVERY OF CIRCLEVILLE,NaN,NaN,0
4,PT100004,65+,M,Central,7.892771e+09,Infectious Disease,7.892771e+09,87070,NaN,Lab,2024-08-23,418.11,NaN,Paid,NaN,TREVISAN,MARIE,0


In [17]:

import pandas as pd

# Assuming df is your main claims dataset
# Load updated dataset with age categories
df = pd.read_csv("unified_claims_model_dataset.csv")

# --- Create fact_claim ---
fact_claim = df[['patient_id', 'provider_id', 'procedure_code', 'date_of_service',
                 'billed_amount', 'paid_amount', 'adjudication_status']].copy()

# Add surrogate key for fact table
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

# # Map patient_sk from dim_patient
# patient_map = dict(zip(dim_patient['patient_id_hashed'], dim_patient['patient_sk']))
# fact_claim['patient_sk'] = fact_claim['patient_id'].map(patient_map)

fact_claim['patient_id_hashed'] = fact_claim['patient_id'].apply(hash_id)
patient_map = dict(zip(dim_patient['patient_id_hashed'], dim_patient['patient_sk']))
fact_claim['patient_sk'] = fact_claim['patient_id_hashed'].map(patient_map)


# Map provider_sk from dim_provider
provider_map = dict(zip(dim_provider['provider_id'], dim_provider['provider_sk']))
fact_claim['provider_sk'] = fact_claim['provider_id'].map(provider_map)

# Map proc_sk from dim_procedure
proc_map = dict(zip(dim_procedure['procedure_code'], dim_procedure['proc_sk']))
fact_claim['proc_sk'] = fact_claim['procedure_code'].map(proc_map)

# Drop original IDs (optional)
fact_claim = fact_claim[['claim_sk', 'patient_sk', 'provider_sk', 'proc_sk',
                         'date_of_service', 'billed_amount', 'paid_amount', 'adjudication_status']]

# Validate
print("✅ fact_claim table created successfully!")
print(fact_claim.head())

# Save to CSV
fact_claim.to_csv('fact_claim.csv', index=False)


✅ fact_claim table created successfully!
   claim_sk  patient_sk  provider_sk   proc_sk date_of_service  billed_amount  \
0         1           1          1.0  74476075      2023-10-13        5944.48   
1         2           2          2.0  69893948      2024-03-23        2014.75   
2         3           3          3.0  35867643      2023-11-23            NaN   
3         4           4          4.0  95044104      2024-12-05         502.79   
4         5           5          5.0  21477593      2024-08-23         418.11   

   paid_amount adjudication_status  
0      5880.39                Paid  
1      1926.35                Paid  
2      5168.72                Paid  
3          NaN      Partially Paid  
4          NaN                Paid  


In [18]:

# Add surrogate key first
fact_claim['claim_sk'] = range(1, len(fact_claim) + 1)

# Now run validation
validation_results = []

# 1. Uniqueness Checks
checks = {
    'dim_patient.patient_sk': dim_patient['patient_sk'].is_unique,
    'dim_provider.provider_sk': dim_provider['provider_sk'].is_unique,
    'dim_procedure.proc_sk': dim_procedure['proc_sk'].is_unique,
    'fact_claim.claim_sk': fact_claim['claim_sk'].is_unique
}
for name, result in checks.items():
    validation_results.append({'Check': f'Unique {name}', 'Status': 'PASS' if result else 'FAIL'})

# 2. Null Checks in Critical Columns
critical_fact_cols = ['claim_sk', 'patient_sk', 'provider_sk', 'proc_sk', 'date_of_service']
for col in critical_fact_cols:
    null_count = fact_claim[col].isnull().sum()
    validation_results.append({'Check': f'Nulls in fact_claim.{col}', 'Status': 'PASS' if null_count == 0 else f'FAIL ({null_count} nulls)'})

# 3. Referential Integrity Checks
invalid_patient_refs = fact_claim[~fact_claim['patient_sk'].isin(dim_patient['patient_sk'])]
validation_results.append({'Check': 'Referential integrity patient_sk', 'Status': 'PASS' if invalid_patient_refs.empty else f'FAIL ({len(invalid_patient_refs)} invalid)'})

invalid_provider_refs = fact_claim[~fact_claim['provider_sk'].isin(dim_provider['provider_sk'])]
validation_results.append({'Check': 'Referential integrity provider_sk', 'Status': 'PASS' if invalid_provider_refs.empty else f'FAIL ({len(invalid_provider_refs)} invalid)'})

invalid_proc_refs = fact_claim[~fact_claim['proc_sk'].isin(dim_procedure['proc_sk'])]
validation_results.append({'Check': 'Referential integrity proc_sk', 'Status': 'PASS' if invalid_proc_refs.empty else f'FAIL ({len(invalid_proc_refs)} invalid)'})

# Export validation report
report_df = pd.DataFrame(validation_results)
report_df.to_csv('validation_report.csv', index=False)

print("✅ Validation completed. Report saved as validation_report.csv")
print(report_df)


✅ Validation completed. Report saved as validation_report.csv
                                  Check              Status
0         Unique dim_patient.patient_sk                PASS
1       Unique dim_provider.provider_sk                PASS
2          Unique dim_procedure.proc_sk                PASS
3            Unique fact_claim.claim_sk                PASS
4          Nulls in fact_claim.claim_sk                PASS
5        Nulls in fact_claim.patient_sk                PASS
6       Nulls in fact_claim.provider_sk    FAIL (867 nulls)
7           Nulls in fact_claim.proc_sk                PASS
8   Nulls in fact_claim.date_of_service                PASS
9      Referential integrity patient_sk                PASS
10    Referential integrity provider_sk  FAIL (867 invalid)
11        Referential integrity proc_sk                PASS


In [29]:

# import pandas as pd

# # Load the dataset
# file_path = "unified_claims_model_dataset.csv"
# df = pd.read_csv(file_path)

# # Identify rows with missing provider_id
# missing_provider_rows = df[df['provider_id'].isnull()]

# # Create synthetic providers grouped by speciality
# synthetic_providers = (
#     missing_provider_rows.groupby('speciality')
#     .size()
#     .reset_index(name='count')
# )

# # Assign surrogate keys for synthetic providers
# synthetic_providers['provider_sk'] = range(100000, 100000 + len(synthetic_providers))
# synthetic_providers['provider_id'] = 'SYNTHETIC'
# synthetic_providers['npi'] = 'UNKNOWN'

# # Prepare dim_provider (existing + synthetic)
# existing_providers = (
#     df.dropna(subset=['provider_id'])[['provider_id', 'speciality', 'npi']]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )
# existing_providers['provider_sk'] = range(1, len(existing_providers) + 1)

# # Combine existing and synthetic providers
# synthetic_dim_provider = synthetic_providers[['provider_sk', 'provider_id', 'speciality', 'npi']]
# dim_provider = pd.concat([existing_providers[['provider_sk', 'provider_id', 'speciality', 'npi']], synthetic_dim_provider], ignore_index=True)

# # Map speciality to synthetic provider_sk for missing rows
# speciality_to_sk = dict(zip(synthetic_providers['speciality'], synthetic_providers['provider_sk']))

# # Create fact_claim table (simplified for this step)
# fact_claim = df.copy()

# # Map provider_sk: if provider_id exists, map from existing_providers; else use synthetic based on speciality
# provider_id_to_sk = dict(zip(existing_providers['provider_id'], existing_providers['provider_sk']))

# fact_claim['provider_sk'] = fact_claim.apply(
#     lambda row: provider_id_to_sk.get(row['provider_id']) if pd.notnull(row['provider_id']) else speciality_to_sk.get(row['speciality']),
#     axis=1
# )

# # Validate that no provider_sk is null now
# null_count_after = fact_claim['provider_sk'].isnull().sum()

# # Save updated tables
# dim_provider.to_csv('dim_provider_imputed.csv', index=False)
# fact_claim.to_csv('fact_claim_imputed.csv', index=False)

# print("✅ Synthetic provider imputation completed.")
# print(f"Total synthetic providers created: {len(synthetic_providers)}")
# print(f"Remaining null provider_sk after imputation: {null_count_after}")
# print("Files saved: dim_provider_imputed.csv, fact_claim_imputed.csv")


✅ Synthetic provider imputation completed.
Total synthetic providers created: 30
Remaining null provider_sk after imputation: 0
Files saved: dim_provider_imputed.csv, fact_claim_imputed.csv
